# v10 Visualisation Figures

- **Fig 5** — Segmentation comparison: CT | GT | Stage 29 (skip) | 9b (no-skip)
- **Fig 6** — Prototype heatmaps per class for Stage 29 on a representative slice
- **Fig 7** — Prototype atlas: L4 nearest training patches (what each prototype represents)

Output: `report/v10/figures/`

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, NUM_CLASSES, LABEL_NAMES
from src.models.proto_seg_net import ProtoSegNet
from src.models.proto_seg_net_v2 import ProtoSegNetV2
from src.metrics.proto_quality import build_prototype_atlas

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
FIG_DIR  = Path('report/v10/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

FOREGROUND  = list(range(1, NUM_CLASSES))
CLASS_NAMES = [LABEL_NAMES[k] for k in range(NUM_CLASSES)]

SEG_COLORS = [
    (0, 0, 0, 0),
    (0.9, 0.1, 0.1, 0.7),
    (0.1, 0.5, 0.9, 0.7),
    (0.1, 0.8, 0.2, 0.7),
    (0.9, 0.6, 0.1, 0.7),
    (0.7, 0.1, 0.8, 0.7),
    (0.9, 0.9, 0.1, 0.8),
    (0.1, 0.9, 0.9, 0.7),
]
SEG_CMAP = mcolors.ListedColormap(SEG_COLORS)
SEG_NORM = mcolors.BoundaryNorm(boundaries=np.arange(-0.5, NUM_CLASSES), ncolors=NUM_CLASSES)
HM_COLORS = ['Reds', 'Blues', 'Greens', 'Oranges', 'Purples', 'YlOrBr', 'GnBu']

plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 9})
print(f'Device: {DEVICE}')

Device: mps


## Load models and data

In [2]:
ckpt29 = torch.load(CKPT_DIR / 'proto_seg_ct_l3l4_warmstart.pth', map_location='cpu', weights_only=False)
m29 = ProtoSegNet(
    n_classes=NUM_CLASSES, proto_levels=[3, 4],
    use_level_attention=ckpt29.get('use_level_attention', False),
).to(DEVICE)
m29.load_state_dict(ckpt29['model_state_dict'])
m29.eval()
print(f'Stage 29: epoch={ckpt29["epoch"]}  val_dice={ckpt29["best_val_dice"]:.4f}')

ckpt9b = torch.load(CKPT_DIR / 'proto_seg_ct_v2_l34.pth', map_location='cpu', weights_only=False)
m9b = ProtoSegNetV2(
    n_classes=NUM_CLASSES, proto_levels=[3, 4],
    use_attention=ckpt9b.get('use_attention', False),
).to(DEVICE)
m9b.load_state_dict(ckpt9b['model_state_dict'])
m9b.eval()
print(f'9b: epoch={ckpt9b["epoch"]}  val_dice={ckpt9b["best_val_dice"]:.4f}')

test_ds  = MMWHSSliceDataset(DATA_DIR, 'ct', 'test',  augment=False, preload=True)
train_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'train', augment=False, preload=True)
print(f'Test: {len(test_ds)} slices  Train: {len(train_ds)} slices')

Stage 29: epoch=10  val_dice=0.8215
9b: epoch=60  val_dice=0.5588


Test: 484 slices  Train: 3389 slices


## Select representative slices

In [3]:
scores = []
for i in range(len(test_ds)):
    lbl = test_ds[i]['label']
    n = sum(1 for k in FOREGROUND if (lbl == k).sum() >= 50)
    scores.append((n, i))
scores.sort(reverse=True)

chosen = []
for _, idx in scores:
    if not chosen or all(abs(idx - c) > 30 for c in chosen):
        chosen.append(idx)
    if len(chosen) == 3:
        break
chosen.sort()
print(f'Selected slices: {chosen}')

slices_data = []
for idx in chosen:
    sample = test_ds[idx]
    img = sample['image'].unsqueeze(0).to(DEVICE)
    lbl = sample['label']
    with torch.no_grad():
        logits29, hm29 = m29(img)
        out9b = m9b(img)
        logits9b = out9b[0]
    pred29 = logits29.argmax(dim=1).squeeze(0).cpu().numpy()
    pred9b = logits9b.argmax(dim=1).squeeze(0).cpu().numpy()
    img_np = sample['image'].squeeze(0).numpy()
    slices_data.append({'img': img_np, 'gt': lbl.numpy(), 'pred29': pred29,
                        'pred9b': pred9b, 'hm29': hm29, 'idx': idx})
    present = [CLASS_NAMES[k] for k in FOREGROUND if (lbl == k).sum() >= 50]
    print(f'  Slice {idx}: {present}')

Selected slices: [305, 336, 367]
  Slice 305: ['LV', 'RV', 'LA', 'RA', 'Myocardium', 'Aorta']
  Slice 336: ['LV', 'RV', 'LA', 'RA', 'Myocardium', 'Aorta']
  Slice 367: ['LV', 'RV', 'RA', 'Myocardium', 'Aorta', 'PA']


## Fig 5 — Segmentation Comparison

In [4]:
col_titles = ['CT Input', 'Ground Truth', 'Stage 29 (skip, L3+L4)', '9b (no-skip, L3+L4)']
fig, axes = plt.subplots(3, 4, figsize=(13, 9.5))

for row, sd in enumerate(slices_data):
    for col, arr in enumerate([sd['img'], sd['gt'], sd['pred29'], sd['pred9b']]):
        ax = axes[row, col]
        ax.imshow(sd['img'], cmap='gray', vmin=-1, vmax=1)
        if col > 0:
            ax.imshow(arr, cmap=SEG_CMAP, norm=SEG_NORM, interpolation='nearest')
        ax.axis('off')
        if row == 0:
            ax.set_title(col_titles[col], fontsize=10, fontweight='bold', pad=5)
        if col == 0:
            ax.set_ylabel(f'Slice {sd["idx"]}', fontsize=9, labelpad=4)

patches = [mpatches.Patch(color=SEG_COLORS[k], label=CLASS_NAMES[k]) for k in FOREGROUND]
fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=9,
           bbox_to_anchor=(0.5, -0.01), framealpha=0.9)
fig.suptitle('Fig 5  |  Segmentation Results: CT | GT | Stage 29 | 9b',
             fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
out = FIG_DIR / 'fig5_segmentation_comparison.png'
fig.savefig(out, bbox_inches='tight', dpi=150)
print(f'Saved: {out}')
plt.close()

Saved: report/v10/figures/fig5_segmentation_comparison.png


## Fig 6 — Per-class Prototype Heatmaps (Stage 29)

In [5]:
sd = slices_data[0]
hm29 = sd['hm29']
img_np = sd['img']
gt_np  = sd['gt']

def build_level_heatmap(hm_dict, level, k, size=(256, 256)):
    A = hm_dict[level]
    A_k = A[0, k, :, :, :].max(dim=0).values
    return F.interpolate(
        A_k.unsqueeze(0).unsqueeze(0), size=size, mode='bilinear', align_corners=False
    ).squeeze().cpu().numpy()

fig, axes = plt.subplots(3, 7, figsize=(16, 6.5))
row_labels = ['L3 (32×32)', 'L4 (16×16)', 'Combined']

for col, (k, cmap) in enumerate(zip(FOREGROUND, HM_COLORS)):
    hm_l3  = build_level_heatmap(hm29, 3, k)
    hm_l4  = build_level_heatmap(hm29, 4, k)
    hm_comb = np.maximum(hm_l3, hm_l4)
    vmax = max(hm_comb.max(), 1e-6)

    for row, hm in enumerate([hm_l3, hm_l4, hm_comb]):
        ax = axes[row, col]
        ax.imshow(img_np, cmap='gray', vmin=-1, vmax=1)
        ax.imshow(hm, cmap=cmap, alpha=0.65, vmin=0, vmax=vmax)
        ax.contour(gt_np == k, levels=[0.5], colors='white', linewidths=0.9, alpha=0.9)
        ax.axis('off')
        if row == 0:
            ax.set_title(CLASS_NAMES[k], fontsize=10, fontweight='bold')

for row, rl in enumerate(row_labels):
    axes[row, 0].set_ylabel(rl, fontsize=9, rotation=90, labelpad=4, va='center')

fig.suptitle(
    f'Fig 6  |  Stage 29 Per-class Prototype Heatmaps (slice {sd["idx"]})  '
    f'— white contour = GT boundary',
    fontsize=11, fontweight='bold', y=1.02
)
fig.tight_layout()
out = FIG_DIR / 'fig6_heatmaps_per_class.png'
fig.savefig(out, bbox_inches='tight', dpi=150)
print(f'Saved: {out}')
plt.close()

Saved: report/v10/figures/fig6_heatmaps_per_class.png


## Fig 7 — Prototype Atlas (Stage 29, L4 and L3)

In [6]:
N_ATLAS = min(400, len(train_ds))
atlas_subset = torch.utils.data.Subset(train_ds, list(range(N_ATLAS)))
atlas_loader = torch.utils.data.DataLoader(atlas_subset, batch_size=16, shuffle=False)
print(f'Building atlas from {N_ATLAS} training slices...')

fig_l4 = build_prototype_atlas(m29, atlas_loader, level=4)
fig_l4.suptitle(
    'Fig 7  |  Prototype Atlas — Stage 29, Level L4 (16×16)\n'
    'Each cell: training patch with highest activation · Red contour = GT boundary',
    fontsize=10, fontweight='bold', y=1.04
)
out = FIG_DIR / 'fig7_prototype_atlas_l4.png'
fig_l4.savefig(out, bbox_inches='tight', dpi=150)
print(f'Saved: {out}')
plt.close(fig_l4)

fig_l3 = build_prototype_atlas(m29, atlas_loader, level=3)
fig_l3.suptitle(
    'Fig 7b  |  Prototype Atlas — Stage 29, Level L3 (32×32)\n'
    'Each cell: training patch with highest activation · Red contour = GT boundary',
    fontsize=10, fontweight='bold', y=1.04
)
out_l3 = FIG_DIR / 'fig7b_prototype_atlas_l3.png'
fig_l3.savefig(out_l3, bbox_inches='tight', dpi=150)
print(f'Saved: {out_l3}')
plt.close(fig_l3)

Building atlas from 400 training slices...


Saved: report/v10/figures/fig7_prototype_atlas_l4.png


Saved: report/v10/figures/fig7b_prototype_atlas_l3.png


## Summary

In [7]:
figs = sorted(FIG_DIR.glob('fig*.png'))
print(f'{len(figs)} figures in {FIG_DIR}:')
for f in figs:
    print(f'  {f.name:45s}  {f.stat().st_size//1024} KB')

7 figures in report/v10/figures:
  fig2_2x2_heatmap.png                           91 KB
  fig3_pixel_vs_patch_faith.png                  75 KB
  fig4_ap_vs_unet.png                            99 KB
  fig5_segmentation_comparison.png               452 KB
  fig6_heatmaps_per_class.png                    1481 KB
  fig7_prototype_atlas_l4.png                    232 KB
  fig7b_prototype_atlas_l3.png                   112 KB
